
# Pickle-only visual identity name QA

This notebook runs the same visual cleaning + identity logic you used, but **without InsightFace** and without needing the `Frames/` folder.

It uses only:
- `data/*_visual.pkl`
- `data/speaker_by_second_all_debates_named_clean.csv`
- `src/visual_cleaning.py`
- `src/visual_identity_solver.py`

It maps `person_A`, `person_B`, `person_C` to real names and gives you a widget/table to check whether the mapping looks correct.


In [12]:

from pathlib import Path
import sys
import importlib
import json
from itertools import permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()

# Works if notebook is inside notebooks/ or project root
if (CURRENT_DIR / "src").exists() and (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists() and (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "pickle_only_name_QA"
PROCESSED_DIR = OUTPUT_DIR / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import visual_identity_solver
import visual_cleaning

importlib.reload(visual_identity_solver)
importlib.reload(visual_cleaning)

solver = visual_identity_solver
from visual_cleaning import clean_visual_dataframe, add_clean_emotion_columns

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Loaded solver from:", visual_identity_solver.__file__)
print("Loaded cleaning from:", visual_cleaning.__file__)


Project root: C:\Users\lucas\Documents\big_data
Data dir: C:\Users\lucas\Documents\big_data\data
Output dir: C:\Users\lucas\Documents\big_data\outputs\pickle_only_name_QA
Loaded solver from: C:\Users\lucas\Documents\big_data\src\visual_identity_solver.py
Loaded cleaning from: C:\Users\lucas\Documents\big_data\src\visual_cleaning.py


In [13]:

# ------------------------------------------------------------
# Load speaker CSV and visual pickle list
# ------------------------------------------------------------

SPEAKER_CSV = DATA_DIR / "speaker_by_second_all_debates_named_clean.csv"

if not SPEAKER_CSV.exists():
    print("Could not find:", SPEAKER_CSV)
    print("CSV files found in data folder:")
    for p in DATA_DIR.rglob("*.csv"):
        print(" -", p)
    raise FileNotFoundError(SPEAKER_CSV)

voice_all = pd.read_csv(SPEAKER_CSV)

visual_pkl_paths = sorted(DATA_DIR.rglob("*_visual.pkl"))

if len(visual_pkl_paths) == 0:
    print("No *_visual.pkl files found in:", DATA_DIR)
    raise FileNotFoundError("No *_visual.pkl files found")

print("Speaker CSV rows:", len(voice_all))
print("Debates in speaker CSV:", voice_all["debate_name"].nunique())
print("Visual pickle files found:", len(visual_pkl_paths))

print("\nFirst visual pickle files:")
for p in visual_pkl_paths[:10]:
    print(" -", p.relative_to(PROJECT_ROOT))


Speaker CSV rows: 57737
Debates in speaker CSV: 28
Visual pickle files found: 28

First visual pickle files:
 - data\raw_features\Cotrim_Figueiredo_vs_Filipe_November_30_visual.pkl
 - data\raw_features\Cotrim_Figueiredo_vs_Gouveia_Melo_November_20_visual.pkl
 - data\raw_features\Cotrim_Figueiredo_vs_Marques_Mendes_December_7_visual.pkl
 - data\raw_features\Cotrim_Figueiredo_vs_Ventura_December_19_visual.pkl
 - data\raw_features\Filipe_vs_Gouveia_Melo_December_2_visual.pkl
 - data\raw_features\Filipe_vs_Marques_Mendes_November_18_visual.pkl
 - data\raw_features\Filipe_vs_Martins_December_10_visual.pkl
 - data\raw_features\Filipe_vs_Pinto_December_8_visual.pkl
 - data\raw_features\Gouveia_Melo_vs_Ventura_December_15_visual.pkl
 - data\raw_features\Marques_Mendes_vs_Gouveia_Melo_December_21_visual.pkl


In [14]:

# ------------------------------------------------------------
# Helper functions for debate-name matching and candidate mapping
# ------------------------------------------------------------

def normalize_text(value):
    """Lowercase and remove non-alphanumeric characters for fuzzy matching."""
    import re
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def match_debate_name_from_pkl(pkl_path, voice_all_df):
    """
    Match a *_visual.pkl filename to the debate_name used in the speaker CSV.
    Example:
        Ventura_vs_Seguro_November_17_visual.pkl -> Ventura vs Seguro
    """
    stem_norm = normalize_text(Path(pkl_path).stem.replace("_visual", ""))

    debate_names = (
        voice_all_df["debate_name"]
        .dropna()
        .drop_duplicates()
        .astype(str)
        .tolist()
    )

    # Longer names first so "Cotrim Figueiredo vs Filipe" wins over a partial shorter match.
    debate_names = sorted(debate_names, key=len, reverse=True)

    for debate_name in debate_names:
        if normalize_text(debate_name) in stem_norm:
            return debate_name

    return None


def candidate_names_for_debate(voice_df):
    """Return the two candidate names from a speaker dataframe for one debate."""
    names = []
    for col in ["candidate_1", "candidate_2"]:
        if col in voice_df.columns:
            values = voice_df[col].dropna().astype(str).unique().tolist()
            names.extend(values)

    names = [n for n in names if n not in ["", "nan", "None"]]
    names = sorted(set(names))
    return names


def mode_or_none(series):
    series = series.dropna()
    series = series[series.astype(str) != ""]
    if len(series) == 0:
        return None
    return series.value_counts().index[0]


In [15]:

# ------------------------------------------------------------
# Pickle-only identity solver: same notebook logic, no InsightFace
# ------------------------------------------------------------

USE_INSIGHTFACE = False
FORCE_RERUN = False


def run_pickle_only_identity_for_pkl(visual_pkl, force_rerun=False):
    """
    Runs your notebook identity logic but with use_insightface=False.
    Uses only the *_visual.pkl features: poses, face boxes, landmarks, emotions.
    """
    visual_pkl = Path(visual_pkl)
    debate_id = visual_pkl.stem.replace("_visual", "")

    solver_output_path = PROCESSED_DIR / f"{debate_id}_pickle_only_predictions.pkl"
    solver_frames_path = PROCESSED_DIR / f"{debate_id}_pickle_only_frames.pkl"

    if solver_output_path.exists() and solver_frames_path.exists() and not force_rerun:
        out = pd.read_pickle(solver_output_path)
        df_solver = pd.read_pickle(solver_frames_path)
        model_name = "cached_pickle_only"
        return out, df_solver, model_name

    cfg = solver.Config(
        pkl=visual_pkl,
        project_root=PROJECT_ROOT,
        frames_root=Path("Frames"),
        out=PROCESSED_DIR / f"{debate_id}_pickle_only_solver_outputs",

        # Important: pickle-only mode
        use_insightface=False,

        # Same settings as your notebook
        force_small_two_has_person2=True,
        use_first_single_p2_prior=False,
        early_single_p2_always=False,
        constraint_first=True,
        prototype_temperature=2.25,
        low_confidence_threshold=0.55,
        two_large_sum_area=0.55,
        two_touching_gap=0.035,
        save_debug_images=False,
        annotate_every=0,
    )

    print("Loading raw pickle:", visual_pkl.name)
    df_raw = solver.load_visual_pickle(visual_pkl)

    print("Cleaning visual detections...")
    df_clean = clean_visual_dataframe(df_raw)
    df_clean = add_clean_emotion_columns(df_clean)

    # The solver expects columns named Poses and Fer.
    # Replace them with cleaned detections, just like your notebook.
    df_solver = df_clean.copy()
    df_solver["Poses"] = df_solver["Clean_Poses"]
    df_solver["Fer"] = df_solver["Clean_Fer"]

    df_solver["frame_num"] = df_solver["Frame"].map(solver.parse_frame_num)
    df_solver = df_solver.sort_values("frame_num").reset_index(drop=True)

    df_solver["n_poses"] = df_solver["Poses"].map(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    df_solver["n_faces"] = df_solver["Fer"].map(
        lambda x: sum(1 for f in x if isinstance(f, dict)) if isinstance(x, list) else 0
    )

    width, height = solver.infer_frame_size(df_solver, PROJECT_ROOT)
    print("Frame size inferred:", width, height)

    print("Building detection table from cleaned boxes...")
    det = solver.build_detection_table(df_solver, cfg, width, height)

    print("Creating pickle-based features...")
    base_X, base_names, aux = solver.create_base_features(det, width, height)
    det = pd.concat([det.reset_index(drop=True), aux.reset_index(drop=True)], axis=1)

    print("Building model features without InsightFace...")
    arc_X = None
    X = solver.build_model_features(det, base_X, arc_X, cfg)

    print("Assigning weak labels...")
    det = solver.assign_weak_labels(det, cfg)

    print("Fitting identity model...")
    if cfg.constraint_first:
        probs, pred, model_name = solver.fit_prototype_identity_model(X, det, cfg)
    else:
        probs, pred, model_name = solver.fit_identity_model(X, det, cfg)

    print("Model:", model_name)

    print("Applying frame constraints...")
    out = solver.apply_frame_constraints(det, probs, cfg)

    # Same correction as your notebook:
    # keep single-person shots as unconstrained model predictions.
    out["unconstrained_model_person"] = out["model_person"]
    out["unconstrained_model_confidence"] = out["model_confidence"]

    single_mask = out["n_faces"] == 1

    out.loc[single_mask, "person_label"] = out.loc[single_mask, "unconstrained_model_person"]
    out.loc[single_mask, "confidence"] = out.loc[single_mask, "unconstrained_model_confidence"]
    out.loc[single_mask, "assignment_source"] = "model_single_unconstrained"

    out["model_person"] = out["person_label"]
    out["model_confidence"] = out["confidence"]

    out.to_pickle(solver_output_path)
    df_solver.to_pickle(solver_frames_path)

    return out, df_solver, model_name


In [16]:

# ------------------------------------------------------------
# Name mapping logic from single-person speaking runs
# ------------------------------------------------------------

MIN_SINGLE_RUN = 10
MIN_DOMINANT_SHARE = 0.60
MIN_MEAN_CONFIDENCE = 0.55
VOICE_TIME_OFFSET = 0


def infer_visual_to_name_mapping(out, voice_df, candidate_names):
    """
    Maps person_A/person_C to candidate names using the same logic used in your notebook:
    - find single-person visual runs of at least 10 seconds
    - check who is speaking during those seconds
    - use reliable runs to vote for person_A and person_C
    - person_B is always Moderador/Other
    """
    single_frames = (
        out.loc[
            out["n_faces"] == 1,
            ["frame", "frame_num", "person_label", "confidence"],
        ]
        .drop_duplicates(subset=["frame_num"])
        .sort_values("frame_num")
        .reset_index(drop=True)
    )

    if len(single_frames) == 0:
        visual_to_name_map = {
            "person_A": "person_A_UNRESOLVED",
            "person_B": "Moderador/Other",
            "person_C": "person_C_UNRESOLVED",
        }
        return visual_to_name_map, pd.DataFrame(), pd.DataFrame()

    single_frames["voice_second"] = single_frames["frame_num"].astype(int) + VOICE_TIME_OFFSET

    single_frames = single_frames.merge(
        voice_df[["second", "speaker_name", "voice_label", "estimated_speaker"]],
        left_on="voice_second",
        right_on="second",
        how="left",
    )

    single_frames["new_run"] = (
        single_frames["person_label"].ne(single_frames["person_label"].shift())
        | single_frames["frame_num"].diff().fillna(1).ne(1)
    )

    single_frames["run_id"] = single_frames["new_run"].cumsum()

    run_rows = []

    for run_id, g in single_frames.groupby("run_id"):
        g = g.sort_values("frame_num")

        visual_label = g["person_label"].iloc[0]
        run_length = int(len(g))

        if visual_label not in ["person_A", "person_C"]:
            continue

        if run_length < MIN_SINGLE_RUN:
            continue

        start_sec = int(g["frame_num"].min())
        end_sec = int(g["frame_num"].max())
        mean_confidence = float(g["confidence"].mean())

        candidate_speaking = g[g["speaker_name"].isin(candidate_names)]

        if len(candidate_speaking) == 0:
            dominant_speaker = None
            dominant_count = 0
            dominant_share = 0.0
        else:
            counts = candidate_speaking["speaker_name"].value_counts()
            dominant_speaker = counts.index[0]
            dominant_count = int(counts.iloc[0])
            dominant_share = float(dominant_count / run_length)

        run_rows.append(
            {
                "run_id": int(run_id),
                "person_label": visual_label,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "run_length": run_length,
                "mean_confidence": mean_confidence,
                "dominant_speaker": dominant_speaker,
                "dominant_count": dominant_count,
                "dominant_share": dominant_share,
            }
        )

    run_summary = pd.DataFrame(run_rows)

    if len(run_summary) == 0:
        reliable_runs = pd.DataFrame()
    else:
        reliable_runs = run_summary[
            (run_summary["run_length"] >= MIN_SINGLE_RUN)
            & (run_summary["dominant_share"] >= MIN_DOMINANT_SHARE)
            & (run_summary["mean_confidence"] >= MIN_MEAN_CONFIDENCE)
            & (run_summary["dominant_speaker"].notna())
        ].copy()

    if len(reliable_runs) > 0:
        mapping_votes = (
            reliable_runs
            .groupby(["person_label", "dominant_speaker"])
            .agg(
                n_runs=("run_id", "count"),
                total_seconds=("run_length", "sum"),
                mean_run_length=("run_length", "mean"),
                mean_share=("dominant_share", "mean"),
                mean_confidence=("mean_confidence", "mean"),
            )
            .reset_index()
            .sort_values(
                ["person_label", "total_seconds", "n_runs", "mean_share", "mean_confidence"],
                ascending=[True, False, False, False, False],
            )
        )
    else:
        mapping_votes = pd.DataFrame(
            columns=[
                "person_label",
                "dominant_speaker",
                "n_runs",
                "total_seconds",
                "mean_run_length",
                "mean_share",
                "mean_confidence",
            ]
        )

    score_table = {
        (row["person_label"], row["dominant_speaker"]): float(row["total_seconds"])
        for _, row in mapping_votes.iterrows()
    }

    best_score = -1
    best_assignment = {}

    if len(candidate_names) >= 2:
        for perm in permutations(candidate_names, 2):
            candidate_assignment = {
                "person_A": perm[0],
                "person_C": perm[1],
            }

            score = sum(
                score_table.get((visual_label, speaker_name), 0)
                for visual_label, speaker_name in candidate_assignment.items()
            )

            if score > best_score:
                best_score = score
                best_assignment = candidate_assignment

    visual_to_name_map = {
        "person_A": best_assignment.get("person_A", "person_A_UNRESOLVED"),
        "person_B": "Moderador/Other",
        "person_C": best_assignment.get("person_C", "person_C_UNRESOLVED"),
    }

    return visual_to_name_map, run_summary, mapping_votes


In [ ]:

# ------------------------------------------------------------
# Choose one debate and run pickle-only QA
# ------------------------------------------------------------

pkl_options = []
for p in visual_pkl_paths:
    matched_name = match_debate_name_from_pkl(p, voice_all)
    label = f"{p.name}"
    if matched_name is not None:
        label = f"{matched_name}  |  {p.name}"
    pkl_options.append((label, str(p)))

pkl_dropdown = widgets.Dropdown(
    options=pkl_options,
    description="Debate:",
    layout=widgets.Layout(width="95%"),
)

force_checkbox = widgets.Checkbox(
    value=False,
    description="Force rerun identity solver",
)

run_button = widgets.Button(
    description="Run selected debate",
    button_style="success",
)

run_output = widgets.Output()

# Globals filled after clicking the button
out = None
df_solver = None
out_named = None
voice_df = None
visual_to_name_map = None
run_summary = None
mapping_votes = None
selected_debate_name = None
selected_visual_pkl = None


def run_selected_debate(_):
    global out, df_solver, out_named, voice_df, visual_to_name_map
    global run_summary, mapping_votes, selected_debate_name, selected_visual_pkl

    with run_output:
        clear_output(wait=True)

        selected_visual_pkl = Path(pkl_dropdown.value)
        selected_debate_name = match_debate_name_from_pkl(selected_visual_pkl, voice_all)

        print("Selected pickle:", selected_visual_pkl)
        print("Matched speaker CSV debate_name:", selected_debate_name)

        if selected_debate_name is None:
            print("Could not match this pickle to a debate_name in the speaker CSV.")
            return

        voice_df = (
            voice_all[voice_all["debate_name"] == selected_debate_name]
            .copy()
            .sort_values("second")
            .reset_index(drop=True)
        )

        candidates = candidate_names_for_debate(voice_df)
        print("Candidates:", candidates)
        print("Voice rows:", len(voice_df))

        out, df_solver, model_name = run_pickle_only_identity_for_pkl(
            selected_visual_pkl,
            force_rerun=force_checkbox.value,
        )

        visual_to_name_map, run_summary, mapping_votes = infer_visual_to_name_mapping(
            out,
            voice_df,
            candidates,
        )

        out_named = out.copy()
        out_named["display_label"] = (
            out_named["person_label"]
            .map(visual_to_name_map)
            .fillna(out_named["person_label"])
        )
        out_named["second"] = out_named["frame_num"].astype(int)

        print("\nModel:", model_name)
        print("\nFinal mapping:")
        display(pd.DataFrame(
            list(visual_to_name_map.items()),
            columns=["visual_label", "display_name"],
        ))

        print("\nMapping votes:")
        display(mapping_votes)

        print("\nSingle-person runs used for mapping / review:")
        display(run_summary.head(30))

        mapping_path = OUTPUT_DIR / "selected_debate_candidate_name_mapping_PICKLE_ONLY.csv"
        pd.DataFrame(
            list(visual_to_name_map.items()),
            columns=["visual_label", "display_name"],
        ).assign(
            debate_name=selected_debate_name,
            visual_pkl=selected_visual_pkl.name,
        ).to_csv(mapping_path, index=False)

        print("\nSaved selected debate mapping to:", mapping_path)
        print("\nNow run the next cell to open the QA widget.")

run_button.on_click(run_selected_debate)

display(widgets.VBox([pkl_dropdown, force_checkbox, run_button, run_output]))


In [20]:

# ------------------------------------------------------------
# QA widget: verify if names are being added correctly
# ------------------------------------------------------------

if out_named is None or voice_df is None:
    print("Run the previous cell first and click 'Run selected debate'.")
else:
    voice_lookup = voice_df.set_index("second")["speaker_name"].to_dict()

    qa_seconds = sorted(out_named["second"].dropna().astype(int).unique().tolist())

    qa_slider = widgets.IntSlider(
        value=qa_seconds[0],
        min=min(qa_seconds),
        max=max(qa_seconds),
        step=1,
        description="Second",
        continuous_update=False,
        layout=widgets.Layout(width="95%"),
    )

    qa_jump = widgets.IntText(
        value=qa_seconds[0],
        description="Go to sec:",
    )

    qa_jump_button = widgets.Button(
        description="Jump",
        button_style="info",
    )

    only_single_checkbox = widgets.Checkbox(
        value=False,
        description="Jump only to single-person frames",
    )

    qa_output = widgets.Output()

    def qa_nearest_second(target):
        if only_single_checkbox.value:
            candidates = sorted(
                out_named.loc[out_named["n_faces"] == 1, "second"]
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )
            if len(candidates) == 0:
                candidates = qa_seconds
        else:
            candidates = qa_seconds

        return min(candidates, key=lambda x: abs(x - target))

    def jump_to_qa_second(_):
        qa_slider.value = qa_nearest_second(int(qa_jump.value))

    qa_jump_button.on_click(jump_to_qa_second)

    def show_qa_second(change=None):
        sec = int(qa_slider.value)

        with qa_output:
            clear_output(wait=True)

            frame_dets = (
                out_named[out_named["second"] == sec]
                .sort_values("face_cx")
                .copy()
            )

            csv_speaker = voice_lookup.get(sec, "Unknown")

            print("Debate:", selected_debate_name)
            print("Second:", sec)
            print("Speaker from CSV:", csv_speaker)
            print("Detections in visual pickle:", len(frame_dets))

            if len(frame_dets) == 0:
                print("No visual detections at this second.")
                return

            display_cols = [
                "second",
                "face_idx_lr",
                "person_label",
                "display_label",
                "confidence",
                "n_faces",
                "n_poses",
                "assignment_source",
                "top_emotion",
                "face_area_norm",
            ]
            display_cols = [c for c in display_cols if c in frame_dets.columns]
            display(frame_dets[display_cols])

            print("\nInterpretation check:")
            if len(frame_dets) == 1:
                detected_name = str(frame_dets.iloc[0]["display_label"])
                detected_label = str(frame_dets.iloc[0]["person_label"])
                print("Single-person frame.")
                print("Visual identity:", detected_label, "->", detected_name)
                print("CSV speaker:", csv_speaker)
                if detected_name == csv_speaker:
                    print("✅ Visual name matches CSV speaker at this second.")
                elif csv_speaker in ["No speech", "Moderador/Other", "Unknown"]:
                    print("ℹ️ CSV speaker is not a candidate speaking moment.")
                else:
                    print("⚠️ Visual name does not match CSV speaker. Check nearby seconds too.")
            else:
                print("Multi-person frame. Use this mostly to check labels/display names, not direct speaking match.")

    qa_slider.observe(show_qa_second, names="value")

    display(
        widgets.VBox(
            [
                widgets.HTML("<h3>Pickle-only name mapping QA widget</h3>"),
                qa_slider,
                widgets.HBox([qa_jump, qa_jump_button, only_single_checkbox]),
                qa_output,
            ]
        )
    )

    show_qa_second()


In [21]:

# ------------------------------------------------------------
# Optional: run all debates and export only the candidate-name mapping CSV
# ------------------------------------------------------------

RUN_ALL_DEBATES = False
FORCE_RERUN_ALL = False

if RUN_ALL_DEBATES:
    all_mapping_rows = []

    for i, visual_pkl in enumerate(visual_pkl_paths, start=1):
        print(f"\n[{i}/{len(visual_pkl_paths)}] {visual_pkl.name}")

        debate_name = match_debate_name_from_pkl(visual_pkl, voice_all)

        if debate_name is None:
            print("  Could not match debate name. Skipping.")
            continue

        debate_voice_df = (
            voice_all[voice_all["debate_name"] == debate_name]
            .copy()
            .sort_values("second")
            .reset_index(drop=True)
        )

        candidates = candidate_names_for_debate(debate_voice_df)

        try:
            debate_out, debate_df_solver, model_name = run_pickle_only_identity_for_pkl(
                visual_pkl,
                force_rerun=FORCE_RERUN_ALL,
            )

            debate_map, debate_run_summary, debate_votes = infer_visual_to_name_mapping(
                debate_out,
                debate_voice_df,
                candidates,
            )

            for visual_label, display_name in debate_map.items():
                all_mapping_rows.append(
                    {
                        "debate_name": debate_name,
                        "visual_pkl": visual_pkl.name,
                        "visual_label": visual_label,
                        "candidate": display_name,
                        "is_unresolved": int("UNRESOLVED" in str(display_name)),
                        "n_mapping_runs": int(
                            len(debate_run_summary[debate_run_summary["person_label"] == visual_label])
                            if len(debate_run_summary) > 0 and "person_label" in debate_run_summary.columns
                            else 0
                        ),
                        "model_name": model_name,
                        "use_insightface": False,
                    }
                )

            print("  Mapping:", debate_map)

        except Exception as e:
            print("  ERROR:", repr(e))
            all_mapping_rows.append(
                {
                    "debate_name": debate_name,
                    "visual_pkl": visual_pkl.name,
                    "visual_label": "ERROR",
                    "candidate": repr(e),
                    "is_unresolved": 1,
                    "n_mapping_runs": 0,
                    "model_name": "ERROR",
                    "use_insightface": False,
                }
            )

    all_mapping_df = pd.DataFrame(all_mapping_rows)
    all_mapping_path = OUTPUT_DIR / "01_candidate_name_mapping_all_debates_PICKLE_ONLY.csv"
    all_mapping_df.to_csv(all_mapping_path, index=False)

    print("\nSaved all-debates mapping CSV:")
    print(all_mapping_path)
    display(all_mapping_df)
else:
    print("RUN_ALL_DEBATES is False. Change it to True in this cell to export the all-debates mapping CSV.")


RUN_ALL_DEBATES is False. Change it to True in this cell to export the all-debates mapping CSV.


In [22]:

# Check whether actual frame images are available

sample_frames = df_solver["Frame"].head(20).tolist()

for frame_value in sample_frames:
    path = solver.resolve_frame_path(
        PROJECT_ROOT,
        Path("Frames"),
        frame_value,
    )

    print("Frame value:", frame_value)
    print("Resolved path:", path)
    print("Exists:", path.exists() if path is not None else False)
    print("-" * 60)


Frame value: Frames/Cotrim_Figueiredo_vs_Filipe_November_30/frame_001.jpg
Resolved path: C:\Users\lucas\Documents\big_data\Frames\Cotrim_Figueiredo_vs_Filipe_November_30\frame_001.jpg
Exists: True
------------------------------------------------------------
Frame value: Frames/Cotrim_Figueiredo_vs_Filipe_November_30/frame_002.jpg
Resolved path: C:\Users\lucas\Documents\big_data\Frames\Cotrim_Figueiredo_vs_Filipe_November_30\frame_002.jpg
Exists: True
------------------------------------------------------------
Frame value: Frames/Cotrim_Figueiredo_vs_Filipe_November_30/frame_003.jpg
Resolved path: C:\Users\lucas\Documents\big_data\Frames\Cotrim_Figueiredo_vs_Filipe_November_30\frame_003.jpg
Exists: True
------------------------------------------------------------
Frame value: Frames/Cotrim_Figueiredo_vs_Filipe_November_30/frame_004.jpg
Resolved path: C:\Users\lucas\Documents\big_data\Frames\Cotrim_Figueiredo_vs_Filipe_November_30\frame_004.jpg
Exists: True
-----------------------------